# HTML and Webscraping

In [1]:
import pandas as pd

## Scraping an HTML table

In [2]:
import requests
from bs4 import BeautifulSoup
from io import StringIO

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2025). We'll use Beautiful Soup to scrape information from this table.

1\. Read in the HTML from the URL using the `requests` library.

In [4]:
url = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"

response = requests.get(url)
response.raise_for_status()

html = response.text

HTTPError: 403 Client Error: Forbidden for url: https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population

2\. Use Beautiful Soup to parse this string into a tree called `soup`

In [ ]:
soup = BeautifulSoup(html, "html.parser")

3\. Determine how many tables are in `soup`. (Hint: use `find_all("table")`.)

In [ ]:
tables = soup.find_all("table")
len(tables)

4\. There are several tables included in `soup`, so we need to narrow it down. Go to the cities table Wikipedia page and "Inspect" it. What are the attributes (class, style) of this table?



In [ ]:
# Attributes of the U.S. cities table found by inspecting the page
city_table_attributes = {
    "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
    "style": "text-align:right"
}

city_table_attributes

5\. You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center" style="text-align:right">
```

How many tables in `soup` have these attributes?

In [ ]:
matching_tables = soup.find_all(
    "table",
    attrs={
        "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
        "style": "text-align:right"
    }
)

len(matching_tables)

6\. There should only be 1 table of this type, so we just need to select it. The following code finds all tables with the desired attributes and then selects the first (only) one to store as `table`. (You just need to run this.)

In [ ]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

7\. Our goal is now to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for:

- city
- state
- population (2025 estimate)
- 2020 land area (sq mi).

First, let's just see how to scrape the information for New York City. Starting from `table` create an object, called `city`, that contains the information just for New York City.

Hints: Inspect the source; what kind of tag represents each row? Find all tags of this type in `table` and select the first one that corresponds to a city. Note that the first 3 rows of the table are headers.

In [ ]:
# The first 3 <tr> elements are header rows, so index 3 is New York City.
city = table.find_all("tr")[3]
city

8\. Starting with `city` extract the city's name and store it as `name`.

Hints: Inspect the source; what tag represents the cells within a row? Find all tags of this type and extract the text corresponding to the cell with the city's name.

In [ ]:
cells = city.find_all(["th", "td"])
name = cells[0].get_text(strip=True)
name

9\. Extract the city's state and store it as `state`.

In [ ]:
state = cells[1].get_text(strip=True)
state

10\. Extract the city's population and store is as `population`.

In [ ]:
population = cells[2].get_text(strip=True)
population

11\. Extract the city's area and store is as `area`.

In [ ]:
area = cells[5].get_text(strip=True)
area

12\. Now put the steps for a single city into a loop to extract the information for all cities in `table` and create a data frame.

Hints:
- Start with an empty list named `rows`
- Write a loop that starts `for city in ...` and replace `...` code that finds all the table rows. (Use what you did in part 7, but don't just select one row. Select all rows except for the 3 header rows.)
- Use your code from 8-11 to extract the information for the city
- And append it to `rows` as "name", "state", "population", "area"
- Convert `rows` into a Pandas data frame. You should obtain a data frame with 348 rows and 4 columns.


In [ ]:
rows = []

# Skip the first 3 header rows.
for city in table.find_all("tr")[3:]:
    cells = city.find_all(["th", "td"])

    # Only keep actual city rows.
    if len(cells) >= 6:
        name = cells[0].get_text(strip=True)
        state = cells[1].get_text(strip=True)
        population = cells[2].get_text(strip=True)
        area = cells[5].get_text(strip=True)

        rows.append({
            "name": name,
            "state": state,
            "population": population,
            "area": area
        })

cities = pd.DataFrame(rows)

print(cities.shape)
cities

13\. Use the Pandas command `pd.read_html` can be used to scrape the table from the webpage. Note: `read_html` will return all the tables, so you will need to narrow your request using attributes. You don't need to worry about selecting columns; just scrape the whole table.

In [ ]:
# Use the same table attributes to tell read_html which table we want.
cities_read_html = pd.read_html(
    StringIO(response.text),
    attrs={
        "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
        "style": "text-align:right"
    }
)[0]

cities_read_html

## Scraping from multiple webpages

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/. Notice that the information is spread over many pages.

1\. Scrape the information from the first page with Beatiful Soup.

In [ ]:
hockey_url = "https://www.scrapethissite.com/pages/forms/"

response = requests.get(hockey_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

2\. Find the main table on this page and store it as `table`.

In [ ]:
table = soup.find("table", attrs={"class": "table"})
table

3\. Extract the information from the cells of this table into a Pandas data frame.

In [ ]:
# Get column names from the table header.
headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]

rows = []
for row in table.find_all("tr"):
    cells = row.find_all("td")
    if cells:
        rows.append([cell.get_text(" ", strip=True) for cell in cells])

hockey_page1 = pd.DataFrame(rows, columns=headers)

print(hockey_page1.shape)
hockey_page1

4\. But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We could switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [ ]:
pagination = soup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [ ]:
for link in links:
  print(link.attrs["href"])

Now we can loop over `links` to make a request to the url for each page and scrape the data into a table similar to what we did for the first page. Write such a loop to extract the data and create a Pandas data frame.


A few technicalities:

- You might need to skip the "previous" and "next" buttons
- So you don't keep repeating headers, you will want to skip rows that don't represent teams.

In [ ]:
from urllib.parse import urljoin
import time

base_url = "https://www.scrapethissite.com"

# Get each numbered page link. Using a set prevents duplicate page URLs.
page_links = []
for link in links:
    href = link.get("href")
    text = link.get_text(strip=True)

    # Keep numbered page links and skip Previous/Next.
    if href and text.isdigit():
        page_links.append(urljoin(base_url, href))

page_links = sorted(set(page_links))

all_rows = []

for page_url in page_links:
    page_response = requests.get(page_url)
    page_response.raise_for_status()
    page_soup = BeautifulSoup(page_response.text, "html.parser")
    page_table = page_soup.find("table", attrs={"class": "table"})

    for row in page_table.find_all("tr"):
        cells = row.find_all("td")

        # Header rows contain no <td> cells, so this keeps team rows only.
        if cells:
            all_rows.append([
                cell.get_text(" ", strip=True)
                for cell in cells
            ])

    time.sleep(0.25)

hockey_all_pages = pd.DataFrame(all_rows, columns=headers)

print(hockey_all_pages.shape)
hockey_all_pages